# BAL Score Changes, Teacher Comparison, and Regional Matched Student Counts for Cambridge TKT-Based Professional Development Program Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, inspecting, and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.f74k-xed8/fair2.json`

### Important note about this dataset
The provided Croissant package metadata indicates that the dataset currently exposes **no explicit `cr:RecordSet` entries** (`recordSet: []`). Because all data entities must be referenced by their `@id`, this notebook first inspects the metadata and then conditionally loads records only if record sets are present in the resolved Croissant schema.

This makes the notebook robust: it works both for metadata-only Croissant descriptions and for full Croissant datasets with record sets.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load the dataset metadata from the Croissant schema URL and inspect the top-level dataset information.

Per the notebook requirements, `dataset.metadata` is treated as a single object and not accessed by subscripting.

In [ ]:
import json
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.f74k-xed8/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded successfully.")
print("Name:", getattr(metadata, "name", "N/A"))
print("Description:", getattr(metadata, "description", "N/A"))
print("Version:", getattr(metadata, "version", "N/A"))
print("License:", getattr(metadata, "license", "N/A"))
print("Citation:", getattr(metadata, "cite_as", getattr(metadata, "citeAs", "N/A")))

In [ ]:
# Convert metadata to a JSON-serializable structure for inspection when needed
metadata_json = metadata.to_json()

print("Top-level metadata keys:")
print(sorted(metadata_json.keys()))

## 2. Data Overview
Review the Croissant metadata structure, with special attention to `@id` values for dataset entities.

Because all entities must be referenced by `@id`, the cells below extract any available record set IDs and field IDs directly from the metadata.

In [ ]:
# Basic dataset overview from metadata JSON
dataset_id = metadata_json.get("@id", "N/A")
distribution_entries = metadata_json.get("distribution", [])
record_set_entries = metadata_json.get("recordSet", [])

print("Dataset @id:", dataset_id)
print("Number of distribution entries:", len(distribution_entries))
print("Number of record sets declared:", len(record_set_entries))

if distribution_entries:
    print("\nDistribution @id values:")
    for dist in distribution_entries:
        print("-", dist.get("@id", "N/A"))

if not record_set_entries:
    print("\nNo record sets were explicitly declared in the provided metadata.")

In [ ]:
# Extract record set and field IDs if available
record_set_ids = []
record_set_field_map = {}

for rs in record_set_entries:
    rs_id = rs.get("@id", "N/A")
    record_set_ids.append(rs_id)
    field_entries = rs.get("field", [])
    field_ids = [field.get("@id", "N/A") for field in field_entries]
    record_set_field_map[rs_id] = field_ids

print("Record set IDs:")
print(record_set_ids)

if record_set_field_map:
    print("\nField IDs by record set:")
    for rs_id, field_ids in record_set_field_map.items():
        print(f"- {rs_id}: {field_ids}")

### Metadata summary table
Even when no tabular record sets are available, the Croissant metadata itself contains rich structured information that can be summarized in a DataFrame.

In [ ]:
summary_rows = [
    {"entity_type": "Dataset", "@id": metadata_json.get("@id", "N/A"), "name": metadata_json.get("name", "N/A")},
]

for dist in distribution_entries:
    summary_rows.append({
        "entity_type": "Distribution",
        "@id": dist.get("@id", "N/A"),
        "name": dist.get("name", "")
    })

for rs in record_set_entries:
    summary_rows.append({
        "entity_type": "RecordSet",
        "@id": rs.get("@id", "N/A"),
        "name": rs.get("name", "")
    })
    for field in rs.get("field", []):
        summary_rows.append({
            "entity_type": "Field",
            "@id": field.get("@id", "N/A"),
            "name": field.get("name", "")
        })

metadata_entities_df = pd.DataFrame(summary_rows)
metadata_entities_df

## 3. Data Extraction
This section attempts to load records from each record set using their `@id` values.

Since the provided metadata currently lists no record sets, the code below is written defensively:
- if record sets are available, it loads them;
- otherwise, it creates a metadata-based analysis table from dataset-level information.

In [ ]:
dataframes = {}

if record_set_ids:
    for record_set_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(dataframes[record_set_id])} records from record set: {record_set_id}")
        except Exception as e:
            print(f"Could not load record set {record_set_id}: {e}")
else:
    print("No record sets available for direct record extraction.")

In [ ]:
# Build a dataset-level table from metadata for exploration when no record sets are present
dataset_level_data = {
    "dataset_@id": [metadata_json.get("@id")],
    "name": [metadata_json.get("name")],
    "description": [metadata_json.get("description")],
    "identifier": [metadata_json.get("identifier")],
    "version": [metadata_json.get("version")],
    "datePublished": [metadata_json.get("datePublished")],
    "spatialCoverage": [metadata_json.get("spatialCoverage")],
    "temporalCoverage": [metadata_json.get("temporalCoverage")],
    "citeAs": [metadata_json.get("citeAs")],
    "license": [metadata_json.get("license")],
    "distribution_@id": [distribution_entries[0].get("@id") if distribution_entries else None],
    "n_keywords": [len(metadata_json.get("keywords", []))],
    "n_dataUseCases": [len(metadata_json.get("dataUseCases", []))],
    "n_dataBiases": [len(metadata_json.get("dataBiases", []))],
    "n_dataLimitations": [len(metadata_json.get("dataLimitations", []))],
    "n_dataPreprocessingSteps": [len(metadata_json.get("dataPreprocessingProtocol", []))],
    "n_dataCollectionTimepoints": [len(metadata_json.get("dataCollectionTimeframe", []))]
}

fallback_record_set_id = "dataset_metadata_summary"
dataframes[fallback_record_set_id] = pd.DataFrame(dataset_level_data)

print("Available analysis tables:")
print(list(dataframes.keys()))

dataframes[fallback_record_set_id].head()

### Long-form metadata tables for repeated elements
The dataset metadata contains several repeated lists, such as keywords, biases, limitations, and preprocessing steps. These can be transformed into analysis-friendly DataFrames.

In [ ]:
keywords_df = pd.DataFrame({
    "dataset_@id": [dataset_id] * len(metadata_json.get("keywords", [])),
    "keyword": metadata_json.get("keywords", [])
})

data_biases_df = pd.DataFrame({
    "dataset_@id": [dataset_id] * len(metadata_json.get("dataBiases", [])),
    "dataBias": metadata_json.get("dataBiases", [])
})

data_limitations_df = pd.DataFrame({
    "dataset_@id": [dataset_id] * len(metadata_json.get("dataLimitations", [])),
    "dataLimitation": metadata_json.get("dataLimitations", [])
})

preprocessing_steps_df = pd.DataFrame({
    "dataset_@id": [dataset_id] * len(metadata_json.get("dataPreprocessingProtocol", [])),
    "preprocessing_step": metadata_json.get("dataPreprocessingProtocol", [])
})

timeframe_df = pd.DataFrame({
    "dataset_@id": [dataset_id] * len(metadata_json.get("dataCollectionTimeframe", [])),
    "timepoint": metadata_json.get("dataCollectionTimeframe", [])
})

print("keywords_df shape:", keywords_df.shape)
print("data_biases_df shape:", data_biases_df.shape)
print("data_limitations_df shape:", data_limitations_df.shape)
print("preprocessing_steps_df shape:", preprocessing_steps_df.shape)
print("timeframe_df shape:", timeframe_df.shape)

In [ ]:
keywords_df.head()

## 4. Exploratory Data Analysis (EDA)
Because no tabular record set is exposed in the supplied Croissant metadata, this EDA focuses on structured metadata-derived tables.

We will:
1. quantify list-based metadata fields,
2. normalize a numeric summary field,
3. inspect text lengths for limitations and preprocessing steps,
4. group repeated metadata records by dataset `@id`.

In [ ]:
# Select a numeric field for analysis from the metadata-derived summary table
record_set_id = fallback_record_set_id
numeric_field = "n_dataLimitations"
threshold = 3

filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

std_value = filtered_df[numeric_field].std()
if pd.notnull(std_value) and std_value != 0:
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / std_value
else:
    filtered_df[f"{numeric_field}_normalized"] = 0.0

print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

group_field = "dataset_@id"
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[[numeric_field]].mean(numeric_only=True)
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())

In [ ]:
# Text-length analysis for metadata-derived tables
if not data_limitations_df.empty:
    data_limitations_df = data_limitations_df.copy()
    data_limitations_df["text_length"] = data_limitations_df["dataLimitation"].str.len()
    q1 = data_limitations_df["text_length"].quantile(0.25)
    q3 = data_limitations_df["text_length"].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    non_outlier_limitations_df = data_limitations_df[
        data_limitations_df["text_length"].between(lower_bound, upper_bound)
    ].copy()
    
    std_len = non_outlier_limitations_df["text_length"].std()
    if pd.notnull(std_len) and std_len != 0:
        non_outlier_limitations_df["text_length_normalized"] = (
            non_outlier_limitations_df["text_length"] - non_outlier_limitations_df["text_length"].mean()
        ) / std_len
    else:
        non_outlier_limitations_df["text_length_normalized"] = 0.0
    
    limitations_grouped = non_outlier_limitations_df.groupby("dataset_@id")[["text_length"]].mean()
    print("Data limitations after outlier filtering:")
    print(non_outlier_limitations_df.head())
    print("\nGrouped mean text length by dataset @id:")
    print(limitations_grouped)
else:
    print("No data limitations available for analysis.")

In [ ]:
# Preprocessing step analysis
if not preprocessing_steps_df.empty:
    preprocessing_steps_df = preprocessing_steps_df.copy()
    preprocessing_steps_df["step_length"] = preprocessing_steps_df["preprocessing_step"].str.len()
    preprocessing_steps_df["contains_release"] = preprocessing_steps_df["preprocessing_step"].str.contains("release", case=False, na=False)
    preprocessing_steps_df["contains_match"] = preprocessing_steps_df["preprocessing_step"].str.contains("match", case=False, na=False)
    print(preprocessing_steps_df)
else:
    print("No preprocessing steps available for analysis.")

## 5. Visualization
Visualize metadata-derived summaries to better understand the structure and documentation richness of the dataset.

In [ ]:
# Bar chart of metadata count fields
count_fields = [
    "n_keywords",
    "n_dataUseCases",
    "n_dataBiases",
    "n_dataLimitations",
    "n_dataPreprocessingSteps",
    "n_dataCollectionTimepoints"
]

counts_long_df = dataframes[fallback_record_set_id][count_fields].T.reset_index()
counts_long_df.columns = ["metric", "count"]

plt.figure(figsize=(10, 5))
sns.barplot(data=counts_long_df, x="metric", y="count", palette="viridis")
plt.title("Counts of Metadata Components")
plt.xlabel("Metadata Metric")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Keyword lengths visualization
if not keywords_df.empty:
    keywords_df = keywords_df.copy()
    keywords_df["keyword_length"] = keywords_df["keyword"].str.len()
    
    plt.figure(figsize=(8, 4))
    sns.histplot(keywords_df["keyword_length"], bins=8, kde=True, color="steelblue")
    plt.title("Distribution of Keyword Lengths")
    plt.xlabel("Keyword Length")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()
else:
    print("No keywords available for visualization.")

In [ ]:
# Limitation text lengths visualization
if not data_limitations_df.empty:
    plt.figure(figsize=(10, 5))
    sns.barplot(
        data=data_limitations_df.reset_index(drop=True).assign(item=lambda df: df.index + 1),
        x="item",
        y="text_length",
        palette="magma"
    )
    plt.title("Text Length of Data Limitations")
    plt.xlabel("Limitation Index")
    plt.ylabel("Character Count")
    plt.tight_layout()
    plt.show()
else:
    print("No data limitations available for visualization.")

In [ ]:
# Timeline-style view of collection timeframe values
if not timeframe_df.empty:
    timeframe_df = timeframe_df.copy()
    timeframe_df["timepoint_dt"] = pd.to_datetime(timeframe_df["timepoint"], errors="coerce")
    timeframe_df = timeframe_df.sort_values("timepoint_dt")
    timeframe_df["order"] = range(1, len(timeframe_df) + 1)
    
    plt.figure(figsize=(9, 3))
    sns.scatterplot(data=timeframe_df, x="timepoint_dt", y="order", s=120, color="darkgreen")
    for _, row in timeframe_df.iterrows():
        plt.text(row["timepoint_dt"], row["order"] + 0.05, row["timepoint"], fontsize=9, ha="center")
    plt.title("Reported Data Collection Timepoints")
    plt.xlabel("Time")
    plt.ylabel("Sequence")
    plt.tight_layout()
    plt.show()
else:
    print("No timeframe information available for visualization.")

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to load and inspect a Croissant-described dataset while consistently referencing entities by `@id`.

### Key observations
- The dataset metadata loads successfully from the provided Croissant URL.
- The dataset-level `@id` is available and was used throughout metadata-derived tables.
- One distribution `@id` is present in the metadata.
- No explicit `cr:RecordSet` entries are currently exposed in the provided metadata, so direct tabular record extraction is not available from the schema as given.
- Even without record sets, the Croissant metadata contains rich structured information about use cases, biases, limitations, preprocessing steps, and collection timepoints.

### Practical takeaway
When working with Croissant datasets, it is good practice to:
1. inspect metadata first,
2. extract and use entity `@id` values,
3. conditionally load record sets when they exist,
4. fall back to metadata-driven analysis when the schema is documentation-rich but record-light.

If a future version of this dataset exposes one or more `cr:RecordSet` objects, the same notebook structure can be reused to load records directly into pandas DataFrames for deeper analysis.